# 02 - Bird flight DMD

This workshop uses a small public Toothless hawk dataset to compare PCA/SVD, FFT, and Dynamic Mode Decomposition (DMD) on animal keypoint motion.

## Learning outcomes

By the end of this notebook, you should be able to:

1. name the main pieces of a DMD fit: state matrix, mode, eigenvalue, frequency, growth/decay, amplitude, and reconstruction
2. explain the different answers from PCA/SVD, FFT, and DMD on the same movement
3. animate the original bird flight, the three main DMD mode pairs overlaid, and a DMD reconstruction overlaid on the data
4. make a compact generative flapping example by extending time beyond the recording while the observed bird holds its last pose
5. say clearly what DMD is not designed to do, especially mode-similarity scoring and joint fitting of multiple different examples

If you have time, start with `01_intro_dmd.ipynb`. It uses toy data where the frequencies and spatial patterns are known, so the DMD vocabulary is less abstract before we reach the animal data.


## Setup

Run the next cell once at the start. It installs the workshop package and downloads the small workshop files. After that, the notebook uses ordinary Python for the DMD story.

In [ ]:
# Run this once at the start of Colab.
%pip install -q git+https://github.com/LydiaFrance/dmd-workshop.git@main

from animal_dmd import setup_workshop
setup_workshop()

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
import morphing_birds as mb
import numpy as np
from birddmd import forecast, reconstruct, run_dmd

from animal_dmd import (
    animate_hawk_svd_mode,
    hold_last_frame,
    load_hawk_motion,
    plot_hawk_wingtip_trace_comparison,
    plot_power_spectrum,
    plot_svd_summary,
    plot_unit_circle_eigenvalues,
    print_dmd_summary,
)

np.set_printoptions(precision=3, suppress=True)


## Workflow map

This notebook is meant to feel like the toy notebook, but with the answer key removed. In `01_intro_dmd.ipynb`, we built the left-arm, shared, and decaying motions ourselves and checked whether DMD recovered them. Here, we use the same questions on real hawk keypoints and then decide what the fitted modes seem to be doing.

We will repeat the same sequence as the toy notebook:

1. load motion shaped `(frames, markers, xyz)`
2. animate the original bird motion before fitting anything
3. flatten each frame into one state vector called `data_matrix`
4. use PCA/SVD to ask: which spatial directions compress the motion?
5. use FFT to ask: which frequencies appear in selected traces?
6. use DMD to ask: which spatial modes come with frequencies and growth/decay?
7. animate the DMD reconstruction and the three main mode pairs in one overlay
8. extend the stabilised fitted motion beyond the observed window
9. finish with limitations, because the annoying parts matter for biological interpretation


## Load the hawk data

The default public sample is a clean averaged flapping-only window from a Harris hawk (called Toothless). The helper below gives us the objects we need for the rest of the notebook:

- `hawk`: the drawing skeleton for animation
- `markers`: the motion array, shaped `(frames, markers, xyz)`
- `times`: the time of each frame, in seconds
- `marker_names`: the marker labels
- `dt`: the time step between frames

In [ ]:
hawk, markers, times, marker_names, dt = load_hawk_motion()

print("Loaded the Toothless flapping dataset")
print(f"{markers.shape[0]} frames, {markers.shape[1]} markers, {markers.shape[2]} coordinates")
print(f"{dt:.4f} seconds between frames")
print(marker_names)

## Animate the raw motion

Start with the data, not the algorithm. The play button should show a repeated wingbeat with smaller tail and body motion.


In [ ]:
fig = mb.animate_plotly(hawk, keypoints_frames=markers)

fig

## Make the analysis matrix

The animation wants `(frames, markers, xyz)`. PCA, FFT, and DMD are easier after each frame is flattened into one **state vector**.

We also subtract the mean pose so the analysis describes motion around the average hawk shape. This is the same shape as the toy notebook: `data_matrix` is `(marker coordinates, frames)`.

In [ ]:
mean_pose = markers.mean(axis=0, keepdims=True)
centred_markers = markers - mean_pose
data_matrix = centred_markers.reshape(centred_markers.shape[0], -1).T

print("markers:", markers.shape)
print("centred_markers:", centred_markers.shape)
print("data_matrix:", data_matrix.shape)

## PCA/SVD: spatial patterns

PCA/SVD finds directions that explain variance. It is excellent for compression: a few components can often describe most of the moving shape.

The catch is that a PCA component does not automatically mean "one biological oscillation". PCA does not attach a frequency, growth rate, or time-evolution rule to the spatial pattern. The time scores can mix several rhythms if that is what best explains variance.


In [ ]:
spatial_patterns, singular_values, temporal_scores = np.linalg.svd(data_matrix, full_matrices=False)
variance_explained = singular_values**2 / np.sum(singular_values**2)

plot_svd_summary(
    times,
    variance_explained,
    temporal_scores,
    n_bars=10,
    title="Spatial compression by SVD",
)
plt.show()

print("First four variance percentages:", np.round(variance_explained[:4] * 100, 1))

SVD also gives spatial directions. For the hawk, a static x-z projection is hard to read, so we animate one component by sweeping its score from negative to positive and back again.

This is the same idea as the toy shape plot: move the mean pose along one SVD spatial direction. SVD has a shape direction and a time score; it still does not attach a frequency or growth rule to that shape.


In [ ]:
display(
    animate_hawk_svd_mode(
        hawk,
        mean_pose,
        spatial_patterns,
        singular_values,
        temporal_scores,
        component=1,
    )
)


## FFT: temporal frequencies

FFT identifies temporal frequencies in one signal, such as vertical wingtip motion. It answers a different question from PCA: "what rhythms are present in this trace?"

The catch is that FFT does not by itself give a whole-body spatial mode. A wingtip trace, tail trace, and body trace can have different-looking spectra, and FFT alone does not tell us which markers move together as one coordinated pattern.


In [ ]:
wing_signal = markers[:, marker_names.index("left_wingtip"), 2]
tail_signal = markers[:, marker_names.index("left_tailtip"), 2]

fft_frequencies_hz = np.fft.rfftfreq(wing_signal.size, d=dt)
wing_power = np.abs(np.fft.rfft(wing_signal - wing_signal.mean())) ** 2
tail_power = np.abs(np.fft.rfft(tail_signal - tail_signal.mean())) ** 2

fig, ax = plt.subplots(figsize=(7, 3))
plot_power_spectrum(ax, fft_frequencies_hz, wing_power, "left wingtip z")
plot_power_spectrum(ax, fft_frequencies_hz, tail_power, "left tailtip z")
ax.set_xlim(0, 30)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Power")
ax.set_title("FFT of two marker coordinates")
ax.legend()
plt.show()

wingbeat_frequency_hz = fft_frequencies_hz[1:][np.argmax(wing_power[1:])]
double_frequency_hz = 2 * wingbeat_frequency_hz
print(f"Dominant left-wingtip frequency: {wingbeat_frequency_hz:.2f} Hz")
print(f"Double-frequency target: {double_frequency_hz:.2f} Hz")

## DMD: spatial modes with temporal dynamics

DMD fits one linear map from frame `k` to frame `k + 1`. This is the extra step beyond PCA/SVD and FFT: it tries to connect a spatial pattern to a simple temporal rule.

For each mode, DMD gives us:

- an **eigenvalue**, whose angle gives frequency and whose radius gives growth/decay
- a **mode**, the coordinated spatial pattern
- an **amplitude**, the contribution of that mode at the start of the analysed window

A useful mental model is: PCA/SVD gives spatial axes, FFT gives frequencies in selected traces, and DMD gives spatial axes that come with frequencies and growth/decay.


In [ ]:
dmd_rank = 6
dmd_result = run_dmd(
    data=markers,
    times=times,
    n_modes=dmd_rank,
    d=2,
    eig_constraints={"conjugate_pairs"},
    average_shape=mean_pose,
    n_markers=markers.shape[1],
    verbose=False,
)
dmd_reconstruction = dmd_result.reconstruction
reconstruction_rmse = np.sqrt(np.mean((markers[1:] - dmd_reconstruction) ** 2))

growth_per_second = dmd_result.eigenvalues.real
frequency_hz = dmd_result.eigenvalues.imag / (2 * np.pi)

print_dmd_summary(
    growth_per_second,
    frequency_hz,
    reconstruction_rmse,
    label=f"rank-{dmd_rank} reconstruction",
)

In [ ]:
per_frame_eigenvalues = np.exp(dmd_result.eigenvalues * dt)
plot_unit_circle_eigenvalues(per_frame_eigenvalues)
plt.show()

The toy notebook made this plot easier to read: points near the unit circle are close to constant-amplitude oscillations, points inside decay, and points outside grow.


## Compare raw motion and DMD reconstruction

A useful first check is whether the chosen rank is enough to reconstruct the observed window. This is the bird-data version of the reconstruction check in the toy notebook: before interpreting any individual pair, make sure the sum of the selected DMD modes gives back the motion we fitted.

The black trace is the original bird motion. The crimson trace is the DMD reconstruction from the fitted mode pairs. If these diverge badly, the model is not rich enough or the data window is not close to the kind of motion DMD can represent.

In [ ]:
mb.animate_plotly_compare(
    hawk,
    keypoints_frames_list=[markers[1:], dmd_reconstruction],
    colours=["black", "crimson"],
)

## Inspect DMD mode pairs

Now we do the bird-data version of the toy pair inspection. In the toy notebook, the pair labels were known in advance: left arm, shared coordination, and right-arm decay. Here, the labels are interpretations of fitted modes, so we keep the language more cautious.

A real oscillation usually appears as a positive-frequency and negative-frequency pair. The flapping window also has a slow near-zero-frequency pair that captures the wingtip span increasing over the analysed window.

We will inspect three pairs:

- the **wingbeat pair**, closest to the strongest wingtip FFT peak
- the **near-double-frequency pair**, closest to twice that peak
- the **wingspan-change pair**, closest to 0 Hz

Be careful with the near-double-frequency pair. It is near twice the wingbeat frequency, but not exactly locked to it in this fit. The honest biological interpretation is: this mode modifies the fitted wingbeat motion, so it is worth looking at, but DMD alone does not tell us whether the source is a strict harmonic, feather/wing mechanics, active control, or something else.

This is where DMD becomes more than "PCA plus a spectrum": one overlay animation shows the coordinated marker motion associated with each frequency-labelled pair. These are partial reconstructions, so they will not visually add up to the full bird in the plot.


In [ ]:
pair_frequencies_hz = np.array([
    dmd_result.pair_frequency(pair_index)
    for pair_index in range(dmd_result.n_pairs)
])

wingbeat_pair = int(np.argmin(np.abs(pair_frequencies_hz - wingbeat_frequency_hz)))
double_frequency_pair = int(np.argmin(np.abs(pair_frequencies_hz - double_frequency_hz)))
wingspan_change_pair = int(np.argmin(np.abs(pair_frequencies_hz - 0.0)))
selected_pairs = {
    "wingbeat": wingbeat_pair,
    "near-double-frequency": double_frequency_pair,
    "wingspan change": wingspan_change_pair,
}

for label, pair_number in selected_pairs.items():
    print(f"{label}: pair {pair_number}, {pair_frequencies_hz[pair_number]:.2f} Hz")

near_double_ratio = pair_frequencies_hz[double_frequency_pair] / pair_frequencies_hz[wingbeat_pair]
print(f"Near-double-frequency / wingbeat ratio: {near_double_ratio:.2f}")

wingbeat_motion = reconstruct(dmd_result, pairs=[wingbeat_pair])
double_frequency_motion = reconstruct(dmd_result, pairs=[double_frequency_pair])
wingspan_change_motion = reconstruct(dmd_result, pairs=[wingspan_change_pair])

left_wingtip_index = marker_names.index("left_wingtip")
right_wingtip_index = marker_names.index("right_wingtip")
wingspan_trace = np.linalg.norm(
    wingspan_change_motion[:, left_wingtip_index, :] - wingspan_change_motion[:, right_wingtip_index, :],
    axis=1,
)
print(
    "Wingspan-change pair span: "
    f"{wingspan_trace[0]:.3f} m -> {wingspan_trace[-1]:.3f} m"
)

print("Three pair reconstructions overlaid (royalblue, darkorange, seagreen)")
display(
    mb.animate_plotly_compare(
        hawk,
        keypoints_frames_list=[
            wingbeat_motion,
            double_frequency_motion,
            wingspan_change_motion,
        ],
        colours=["royalblue", "darkorange", "seagreen"],
        alpha=0.25,
    )
)


## Generative DMD extension

DMD can be evaluated at new time points because each mode has a simple time rule. That lets us make small resynthesis examples from the fitted hawk motion.

We will do this in three steps:

1. upsample the fitted reconstruction inside the real flight window
2. slow the fitted wingbeat by changing its frequency
3. extend a stable version of the oscillatory motion beyond the recording

This is still a fitted linear model, not a physics simulator. Treat it as controlled resynthesis of the learned flapping pattern.

### Upsample the observed flight window

First keep the time span honest: no future yet. We ask DMD for the same fitted reconstruction, but at more time points between the first and last reconstructed frames.

The maths stays visible here. `np.linspace` makes the denser time grid, and `reconstruct(..., times=upsampled_times)` evaluates the DMD time rule on that grid.

In [ ]:
upsample_factor = 3
reconstruction_times = times[1:]
upsampled_times = np.linspace(
    reconstruction_times[0],
    reconstruction_times[-1],
    len(reconstruction_times) * upsample_factor,
)
upsampled_reconstruction = reconstruct(dmd_result, times=upsampled_times)

plot_hawk_wingtip_trace_comparison(
    {
        "Original samples": (
            reconstruction_times,
            markers[1:, left_wingtip_index, 2],
        ),
        "DMD reconstruction": (
            reconstruction_times,
            dmd_reconstruction[:, left_wingtip_index, 2],
        ),
        f"Upsampled ({len(upsampled_times)} frames)": (
            upsampled_times,
            upsampled_reconstruction[:, left_wingtip_index, 2],
        ),
    },
    title=f"Temporal upsampling inside the observed window: {upsample_factor}x resolution",
    styles={
        "Original samples": {
            "marker": "o",
            "linestyle": "None",
            "markersize": 4,
        },
        f"Upsampled ({len(upsampled_times)} frames)": {
            "marker": ".",
            "linewidth": 0.5,
            "alpha": 0.7,
        },
    },
)
plt.show()

print(f"Original reconstruction frames: {len(reconstruction_times)}")
print(f"Upsampled reconstruction frames: {len(upsampled_times)}")

### Slow down the fitted wingbeat

For an oscillatory DMD pair, the imaginary part of the continuous-time eigenvalue gives the angular frequency, $\omega$. Frequency in hertz is $\omega / 2\pi$.

Here we keep the fitted wingbeat shape but ask what happens if the oscillation runs at half speed.

In [ ]:
wingbeat_modes = list(dmd_result.conjugate_pairs[wingbeat_pair])
original_omega = abs(dmd_result.eigenvalues[wingbeat_modes[0]].imag)

normal_wingbeat = forecast(
    dmd_result,
    times,
    pairs=[wingbeat_pair],
    stable=True,
)
slow_wingbeat = forecast(
    dmd_result,
    times,
    mode_indices=wingbeat_modes,
    stable=True,
    modify_eigenvalues={
        "zero_modes": wingbeat_modes,
        "changed_freq": original_omega / 2,
    },
)

print(f"Original wingbeat pair: {original_omega / (2 * np.pi):.2f} Hz")
print(f"Slow synthetic wingbeat: {original_omega / (4 * np.pi):.2f} Hz")

plot_hawk_wingtip_trace_comparison(
    {
        "Normal wingbeat pair": (times, normal_wingbeat[:, left_wingtip_index, 2]),
        "Half-frequency wingbeat": (times, slow_wingbeat[:, left_wingtip_index, 2]),
    },
    title="Frequency modification: normal vs half-speed wingbeat",
)
plt.show()

### Stable eternal flight

The DMD time rule for each mode is roughly:

$$\phi \exp(\lambda t)$$

where $\phi$ is the spatial mode and $\lambda = \sigma + i\omega$ is the continuous-time eigenvalue. The imaginary part, $\omega$, sets the oscillation frequency. The real part, $\sigma$, sets the envelope: positive values grow, negative values decay.

For the long forecast we use `stable=True`. In this helper that keeps the fitted oscillation frequency but removes the fitted growth or decay, so the wingbeat does not explode or fade as we extend time.

We also keep only the two oscillatory pairs: wingbeat and near-double-frequency. The wingspan-change pair is near 0 Hz and captures slow drift over the analysed window; extrapolating it would make the bird slowly morph rather than flap steadily.

In [ ]:
expansion_factor = 3.0
oscillatory_pairs = [wingbeat_pair, double_frequency_pair]
extended_times = np.linspace(
    times[0],
    times[-1] * expansion_factor,
    int(len(times) * expansion_factor),
)
stable_future = forecast(
    dmd_result,
    extended_times,
    pairs=oscillatory_pairs,
    stable=True,
)
held_observed = hold_last_frame(markers, len(stable_future))

plot_hawk_wingtip_trace_comparison(
    {
        "Observed flight": (times, markers[:, left_wingtip_index, 2]),
        "Stable forecast": (extended_times, stable_future[:, left_wingtip_index, 2]),
    },
    title="Extended stable forecast (oscillatory pairs only)",
    end_time=times[-1],
)
plt.show()

print("Observed motion holds on the last pose while the stable forecast keeps flapping.")
display(
    mb.animate_plotly_compare(
        hawk,
        keypoints_frames_list=[held_observed, stable_future],
        colours=["black", "crimson"],
    )
)

## What DMD cannot do

DMD is useful, but it has two limitations that are easy to miss.

### 1. There is no universal mathematical "mode similarity" score

A DMD mode is not a uniquely fixed biological object. Complex modes can be rotated in phase, scaled with the opposite change in amplitude, and paired with their conjugate. Two fits can also choose slightly different bases when the data are noisy or modes are close together.

So we should not tell ourselves that there is one clean mathematical number for "how similar are these two modes?" We can compare derived things under an explicit convention: frequency, reconstructed motion, marker amplitudes, phase after choosing a reference marker, or qualitative animations. Those can be useful, but they are analysis choices rather than a universal DMD truth.

### 2. Do not put multiple different examples into one DMD fit and expect a shared biology model

A DMD fit learns one linear operator and one spectrum for one analysed window. If we concatenate several birds, trials, or behaviours, the fit is forced to use one compromise set of modes, or enough rank to memorise each example separately. Neither outcome gives a clean general model of "bird flapping".

The practical workflow is therefore: fit DMD to one coherent window, inspect whether it reconstructs that window, then compare frequencies, reconstructions, and mode animations across examples with care.

That does not make DMD a waste of time. It is good at producing compact, interpretable summaries of repeated motion, at separating frequency-labelled spatial patterns, and at making controlled resyntheses such as removing a wingbeat pair or slowing the fitted flapping rhythm. It is a lens, not the whole microscope.


## Bring-your-own-data template

Replace `markers` and `times` with arrays from another animal:

- `markers`: shape `(num_timesteps, num_keypoints, 3)`
- `times`: shape `(num_timesteps,)`, seconds
- no missing values in the analysed window

Then rerun the PCA, FFT, and DMD sections. For animation, define a matching `morphing_birds` skeleton before calling `animate_plotly`.

For a more detailed custom skeleton setup, use `00_build_your_own_animal.ipynb`.
